In [2]:
import os
import sys
import time
import json
import random
import datetime
import requests
import psycopg
import urllib.parse
from PIL import Image

USER_AGENTS = ["Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36",
               "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.1 Safari/605.1.15"]

def get_headers():
    """
    Randomly select a User-Agent header from the predefined list. 
    """
    return {'User-Agent': random.choice(USER_AGENTS)}


sys.path.insert(0, os.path.abspath('../shared_models'))
from utils import str_like_query
from models import Team, Teams

In [6]:
params = dict()
with open("../.env", "r") as f:
    for line in f.readlines():
        if data:=line.split("="):
            if len(data)==2:
                params[data[0].strip()] = data[1].strip()
db_conn_string =f"postgresql://{params.get('PGUSER')}:{params.get('PGPASSWORD')}@{params.get('DATABASE_HOST')}:{params.get('DATABASE_PORT')}/{params.get('DATABASE_NAME')}"

In [7]:
my_teams = Teams.get_selected(db_conn_string)
len(my_teams.teams)

all_teams = Teams.get_all(db_conn_string)
len(all_teams.teams)

498

In [8]:
twins = Team.get_team_and_events("Twins", "MLB", db_conn_string)

# Process for Adding Future Events

In [10]:
from psycopg.rows import tuple_row

def team_id_lookup(event_team_obj: dict, all_teams:list[Team], league:str) -> str:
    if "uid" in event_team_obj.keys():
        return [x for x in all_teams if x.uid == event_team_obj["uid"]][0].id
    elif "id" in event_team_obj.keys():
        return [x for x in all_teams if str(x.number) == event_team_obj["id"] and x.league.name==league][0].id

def score_exctract(score):
    """
    ... because god forbid we're consistent in how we represent the score of a game...
    """
    if score:
        if isinstance(score, dict):
            return int(score.get("value"))
        else:
            try:
                return int(score)
            except:
                return None
    else:
        return None


def parse_event_teams(teams: list[dict], all_teams:list[Team], league:str) -> dict:
    """
    Make sure we get the home/away teams correct
    """
    team0_id = team_id_lookup(teams[0], all_teams, league)
    team0_score = score_exctract(teams[0].get("score"))
    team1_id = team_id_lookup(teams[1], all_teams, league)
    team1_score = score_exctract(teams[1].get("score"))
    
    if teams[0]["homeAway"] == "home":
        return {"home": {"id": team0_id, "score": team0_score},
                "away": {"id": team1_id, "score": team1_score}}
    else:
        return {"home": {"id": team1_id,"score": team1_score},
                "away": {"id": team0_id, "score": team0_score}}

def parse_event(event_record: dict, all_teams:list[Team], league:str) -> dict:
    """
    Pull the values I want out of the event record from the API
    """
    _teams = parse_event_teams(event_record['competitions'][0]['competitors'], all_teams, league)
    _date = datetime.datetime.strptime(event_record["date"],"%Y-%m-%dT%H:%MZ")
    if event_record.get("status"):
        _status = event_record['status']['type']['description']
    elif _date > datetime.datetime.now():
        _status = "future"
    else:
        _status = "unknown"
    return {
        "id":event_record["id"],
        "uid":event_record.get("uid"),
        "date":_date,
        "name":event_record["name"],
        "short_name":event_record["shortName"],
        "season":event_record["season"].get("slug"),
        "status":_status,
        "home_team_id":_teams["home"]["id"],
        "away_team_id":_teams["away"]["id"],
        "home_score":_teams["home"]["score"],
        "away_score":_teams["away"]["score"],
    }

def event_to_row(event_record):
    """
    Assmble the table-row and append the postgres ID values for the FK relationships
    """
    return (event_record['id'],
            event_record['uid'],
            event_record['date'],
            event_record['name'],
            event_record['short_name'],
            event_record['season'],
            event_record['status'],
            event_record['home_team_id'],
            event_record['away_team_id'],
            event_record['home_score'],
            event_record['away_score'])

# GET Upcoming Events
event_rows = []
my_team_ids = [x.id for x in my_teams.teams]
for team in my_teams.teams:
    #Get the upcoming Event for the team:
    team_url = f"{team.league.teams_url}/{team.number}"
    r = requests.get(team_url, headers=get_headers(), verify=False)
    if not r.status_code == 200:
        raise ValueError("Unable to extract data from the API")
    try:
        next_event = r.json()['team']['nextEvent'][0]
    except Exception as e:
        print(f"failed to extract next event for {team.name} in {team.league.name} with error: {e}")
        continue
    try:
        event_data = parse_event(next_event, all_teams.teams, team.league.name)
        event_rows.append(event_to_row(event_data))
    except Exception as e:
        print(f"failed to parse next event for {team.name} in {team.league.name} with error: {e}")

    # Extract recent events from the Scores API
    scoreboard_url = team.league.scoreboard_url
    r = requests.get(team.league.scoreboard_url, headers=get_headers(), verify=False)
    if not r.status_code == 200:
        raise ValueError(f"Unable to extract scoreboard URL for {team.league.name}")

    _events = r.json().get('events')
    if _events:
        for event in _events:
            try:
                event_data = parse_event(event, all_teams.teams, team.league.name)
                if event_data['home_team_id'] in my_team_ids or event_data['away_team_id'] in my_team_ids:
                    event_rows.append(event_to_row(event_data))
            except Exception as e:
                print(f"failed to parse next event in {team.league.name} with error: {e}")
            



# Update Event Records

with psycopg.connect(db_conn_string) as conn:
    with conn.cursor(row_factory=tuple_row) as cur:
        cur.executemany(
            """
            INSERT INTO events (
                id, uid, date, name, short_name, season, status,
                home_team_id, away_team_id, home_score, away_score
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO UPDATE
            SET uid = EXCLUDED.uid,
                season = EXCLUDED.season,
                status = EXCLUDED.status,
                home_score = EXCLUDED.home_score,
                away_score = EXCLUDED.away_score
            """,
            event_rows,
        )
    conn.commit()
    print(f"udpated database with {len(event_rows)} records")

failed to extract next event for Vikings in NFL with error: list index out of range
udpated database with 6 records


In [16]:
r1 = requests.get("http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard", headers=get_headers(), verify=False)
r1.status_code

200

In [18]:
len(r1.json()['events'])

15

In [19]:
score_events = [parse_event(x, all_teams.teams, 'MLB') for x in r1.json()['events']]

In [20]:
[x for x in score_events if "MIN" in x['short_name']]

[{'id': '401815972',
  'uid': 's:1~l:10~e:401815972',
  'date': datetime.datetime(2026, 7, 1, 0, 10),
  'name': 'Minnesota Twins at Houston Astros',
  'short_name': 'MIN @ HOU',
  'season': 'regular-season',
  'status': 'Final',
  'home_team_id': 'd330a81e-b081-4ad7-8c55-cb7f036042c5',
  'away_team_id': 'a2023f51-e3eb-4639-8c9a-27c1bad88766',
  'home_score': 6,
  'away_score': 4}]

In [28]:
r2 = requests.get("http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/teams/9/schedule", headers=get_headers(), verify=False)
r2.status_code

200

In [31]:
len(r2.json()['events'])

162

In [32]:
parse_event(r2.json()['events'][0], all_teams.teams, 'MLB')

{'id': '401814692',
 'uid': None,
 'date': datetime.datetime(2026, 3, 26, 19, 5),
 'name': 'Minnesota Twins at Baltimore Orioles',
 'short_name': 'MIN @ BAL',
 'season': None,
 'status': 'unknown',
 'home_team_id': '38d0b7ee-6ab6-4cd3-8c1b-710f86c4b003',
 'away_team_id': 'a2023f51-e3eb-4639-8c9a-27c1bad88766',
 'home_score': 2,
 'away_score': 1}

In [34]:
parsed_events = [parse_event(x, all_teams.teams, "MLB") for x in r2.json()['events']]

In [38]:
future_events = [x for x in parsed_events if x['date'] > datetime.datetime.now()]

In [39]:
future_event_rows = [event_to_row(x) for x in future_events]

In [43]:
future_event_rows[-3:]

[('401817082',
  None,
  datetime.datetime(2026, 9, 26, 0, 10),
  'Texas Rangers at Minnesota Twins',
  'TEX @ MIN',
  None,
  'future',
  'a2023f51-e3eb-4639-8c9a-27c1bad88766',
  '9718b8d2-8748-45f9-925b-8cc313b39b4c',
  None,
  None),
 ('401817097',
  None,
  datetime.datetime(2026, 9, 26, 20, 10),
  'Texas Rangers at Minnesota Twins',
  'TEX @ MIN',
  None,
  'future',
  'a2023f51-e3eb-4639-8c9a-27c1bad88766',
  '9718b8d2-8748-45f9-925b-8cc313b39b4c',
  None,
  None),
 ('401817112',
  None,
  datetime.datetime(2026, 9, 27, 19, 10),
  'Texas Rangers at Minnesota Twins',
  'TEX @ MIN',
  None,
  'future',
  'a2023f51-e3eb-4639-8c9a-27c1bad88766',
  '9718b8d2-8748-45f9-925b-8cc313b39b4c',
  None,
  None)]

In [42]:
with psycopg.connect(db_conn_string) as conn:
    with conn.cursor(row_factory=tuple_row) as cur:
        cur.executemany(
            """
            INSERT INTO events (
                id, uid, date, name, short_name, season, status,
                home_team_id, away_team_id, home_score, away_score
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (id) DO UPDATE
            SET uid = EXCLUDED.uid,
                season = EXCLUDED.season,
                status = EXCLUDED.status,
                home_score = EXCLUDED.home_score,
                away_score = EXCLUDED.away_score
            """,
            future_event_rows,
        )
    conn.commit()
    print(f"udpated database with {len(future_event_rows)} records")

udpated database with 75 records


In [47]:
def build_trmnl_payload(last_events, next_events):
    _last = sorted([x for x in last_events if x is not None],
                   key=lambda x: x['date'])
    _next = sorted([x for x in next_events if x is not None],
                   key=lambda x: x['date'])
    core_payload = {"last": {x:_last[x] for x in range(len(_last))},
                    "next": {x:_next[x] for x in range(len(_next))}}
    return {"merge_variables": core_payload}

events= [{"date":datetime.datetime(2026, 9, 29, 0, 10),
          "name": 'Texas Rangers at Minnesota Twins'},
         {"date": datetime.datetime(2026, 9, 26, 20, 10),
          "name": 'Texas Rangers at Minnesota Twins'},
         {"date": datetime.datetime(2026, 9, 27, 19, 10),
          "name": 'Texas Rangers at Minnesota Twins'}]


#sorted(events, key=lambda x: x['date'])
build_trmnl_payload(events, events)

{'merge_variables': {'last': {0: {'date': datetime.datetime(2026, 9, 26, 20, 10),
    'name': 'Texas Rangers at Minnesota Twins'},
   1: {'date': datetime.datetime(2026, 9, 27, 19, 10),
    'name': 'Texas Rangers at Minnesota Twins'},
   2: {'date': datetime.datetime(2026, 9, 29, 0, 10),
    'name': 'Texas Rangers at Minnesota Twins'}},
  'next': {0: {'date': datetime.datetime(2026, 9, 26, 20, 10),
    'name': 'Texas Rangers at Minnesota Twins'},
   1: {'date': datetime.datetime(2026, 9, 27, 19, 10),
    'name': 'Texas Rangers at Minnesota Twins'},
   2: {'date': datetime.datetime(2026, 9, 29, 0, 10),
    'name': 'Texas Rangers at Minnesota Twins'}}}}

In [74]:
now = datetime.datetime.now()

now.day == 1 and now.hour < 6

False

# NFL
Scores: http://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard

News: http://site.api.espn.com/apis/site/v2/sports/football/nfl/news

All Teams: http://site.api.espn.com/apis/site/v2/sports/football/nfl/teams

Specific Team: http://site.api.espn.com/apis/site/v2/sports/football/nfl/teams/:team

# MLB
Scores: http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard

News: http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/news

All Teams: http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/teams

Specific Team: http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/teams/:team

# Hockey
Scores: http://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard

News: http://site.api.espn.com/apis/site/v2/sports/hockey/nhl/news

All Teams: http://site.api.espn.com/apis/site/v2/sports/hockey/nhl/teams

Specific Team: http://site.api.espn.com/apis/site/v2/sports/hockey/nhl/teams/:team

# NBA
Scores: http://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard

News: http://site.api.espn.com/apis/site/v2/sports/basketball/nba/news

All Teams: http://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams

Specific Team: http://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/:team

# WNBA
Scores: http://site.api.espn.com/apis/site/v2/sports/basketball/wnba/scoreboard

News: http://site.api.espn.com/apis/site/v2/sports/basketball/wnba/news

All Teams: http://site.api.espn.com/apis/site/v2/sports/basketball/wnba/teams

Specific Team: http://site.api.espn.com/apis/site/v2/sports/basketball/wnba/teams/:team

# Women's College Basketball
Scores: http://site.api.espn.com/apis/site/v2/sports/basketball/womens-college-basketball/scoreboard

News: http://site.api.espn.com/apis/site/v2/sports/basketball/womens-college-basketball/news

All Teams: http://site.api.espn.com/apis/site/v2/sports/basketball/womens-college-basketball/teams

Specific Team: http://site.api.espn.com/apis/site/v2/sports/basketball/womens-college-basketball/teams/:team

# Build the teams.json file of the core reference material

In [5]:

# leagues = {
#     "NFL": {"teams_url": "http://site.api.espn.com/apis/site/v2/sports/football/nfl/teams",
#             "scoreboard_url": "http://site.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard"},
#     "MLB": {"teams_url": "http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/teams",
#             "scoreboard_url": "http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard"},
#     "NHL": {"teams_url": "http://site.api.espn.com/apis/site/v2/sports/hockey/nhl/teams",
#             "scoreboard_url": "http://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard"},
#     "NBA": {"teams_url": "http://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams",
#             "scoreboard_url": "http://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard"},
#     "WNBA": {"teams_url": "http://site.api.espn.com/apis/site/v2/sports/basketball/wnba/teams",
#              "scoreboard_url": "http://site.api.espn.com/apis/site/v2/sports/basketball/wnba/scoreboard"},
#     "NCAAW": {"teams_url": "http://site.api.espn.com/apis/site/v2/sports/basketball/womens-college-basketball/teams",
#               "scoreboard_url": "http://site.api.espn.com/apis/site/v2/sports/basketball/womens-college-basketball/scoreboard"}
    
# }

# for league_name, league_obj in leagues.items():
#     #if league_name != "NFL":
#     #    break
#     print(league_name)
#     leagues[league_name]["teams"] = dict()
#     r = requests.get(league_obj["teams_url"], headers=get_headers(), verify=False)
#     for team_obj in r.json()['sports'][0]['leagues'][0]['teams']:
#         team_name = team_obj["team"]["slug"]
        
#         leagues[league_name]["teams"][team_name] = { 
#                 "id": team_obj['team']['id'],
#                 "uid": team_obj['team']['uid'],
#                 "name": team_obj['team']['name'],
#                 "display_name": team_obj['team']['displayName'],
#                 "logo_link": f"./resources/icons/{league_name}/{team_name}.svg"
#             }
    
#     time.sleep(1)

# Download the Icon files and convert to SVG

In [61]:
# from PIL import Image
# from pixels2svg import pixels2svg
# import sys
# import os
# import tempfile

# def png_to_svg(png_path, svg_path):
#     # Validate file existence
#     if not os.path.isfile(png_path):
#         raise FileNotFoundError(f"File not found: {png_path}")

#     try:
#         # Open and preprocess image
#         img = Image.open(png_path).convert("RGBA")
        
#         # Optional: Convert to grayscale or reduce colors for cleaner vectorization
#         img = img.convert("P", palette=Image.ADAPTIVE, colors=8)
        
#         # Save preprocessed image to temporary file
#         with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as temp_file:
#             img.save(temp_file.name)
#             temp_path = temp_file.name
        
#         try:
#             # Use pixels2svg with the temporary file
#             svg_str = pixels2svg(temp_path, as_string=True)
            
#             # Save SVG to file
#             with open(svg_path, "w", encoding="utf-8") as f:
#                 f.write(svg_str)
            
#             print(f"SVG saved to {svg_path}")
            
#         finally:
#             # Clean up temporary file
#             os.unlink(temp_path)

#     except Exception as e:
#         print(f"Error converting PNG to SVG: {e}")
#         sys.exit(1)

# #png_to_svg("test.png", "output.svg")

SVG saved to output.svg


# Team URLs for the team details

In [217]:
# Test Pulling Data & Cache locally for reference
#r = requests.get("http://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/16", headers=get_headers(), verify=False)
#print(r.status_code)

#with open('team_sample2.json', 'w') as f:
#    json.dump(r.json(), f, indent=2)

200


In [6]:
team_sample1 = json.load(open("./team_sample.json", "r"))
team_sample2 = json.load(open("./team_sample2.json", "r"))

In [9]:
my_event = Event.from_team_json(team_sample1)

In [12]:
team_sample1['team'].keys()

dict_keys(['id', 'uid', 'slug', 'location', 'name', 'abbreviation', 'displayName', 'shortDisplayName', 'color', 'alternateColor', 'isActive', 'logos', 'record', 'groups', 'links', 'franchise', 'nextEvent', 'standingSummary'])

In [3]:
team_url = "http://site.api.espn.com/apis/site/v2/sports/baseball/mlb/teams/9"

r = requests.get(team_url, headers=get_headers(), verify=False)
r.status_code

200

In [6]:
team_obj = r.json().get('team')
len(team_obj.get("nextEvent"))

1

In [7]:
team_obj.keys()

dict_keys(['id', 'uid', 'slug', 'location', 'name', 'abbreviation', 'displayName', 'shortDisplayName', 'color', 'alternateColor', 'isActive', 'logos', 'record', 'groups', 'links', 'franchise', 'nextEvent', 'standingSummary'])

In [17]:
for event in team_obj.get("nextEvent"):
    _date = datetime.datetime.strptime(event['date'],"%Y-%m-%dT%H:%MZ")
    _id = event.get("id")
    _name = event.get("name")
    if _date < datetime.datetime.now():
        _score = [(x['team']['displayName'], int(x['score']['value'])) for x in event['competitions'][0]['competitors']]
    else:
        _score = None
    print(_date, _id, _name, _score)
    

2026-06-27 00:10:00 401815915 Colorado Rockies at Minnesota Twins None


In [14]:
# def get_team_data(team_url):
#     r = requests.get(team_url, headers=get_headers(), verify=False)
#     if r.status_code == 200:
#         return r.json()['team']
#     else:
#         return None


def extract_events(team_json):
    if team_json and "nextEvent" in team_json.keys():
        if len(team_json['nextEvent']) == 1:
            _date = datetime.datetime.strptime(team_json['nextEvent'][0]['date'],"%Y-%m-%dT%H:%MZ")
            if _date < datetime.datetime.now():
                try:
                    return (team_json['nextEvent'][0]['id'], 
                            team_json['nextEvent'][0]['name'], 
                            _date, 
                            [(x['team']['displayName'], int(x['score']['value'])) for x in team_json['nextEvent'][0]['competitions'][0]['competitors']])
                except KeyError:
                    return (team_json['nextEvent'][0]['id'], 
                            team_json['nextEvent'][0]['name'], 
                            _date, 
                            None)  
            else:
                return (team_json['nextEvent'][0]['id'], 
                        team_json['nextEvent'][0]['name'], 
                        _date, 
                        None)
    return None

extract_events(team_sample1['team']), extract_events(team_sample2['team'])

(('401810569',
  'Minnesota Timberwolves at Memphis Grizzlies',
  datetime.datetime(2026, 2, 3, 0, 30),
  [('Memphis Grizzlies', 137), ('Minnesota Timberwolves', 128)]),
 ('401810582',
  'Minnesota Timberwolves at Toronto Raptors',
  datetime.datetime(2026, 2, 5, 0, 30),
  None))

In [4]:
(datetime.datetime.now() - datetime.datetime(2026, 2, 5, 0, 30)).days

139

In [325]:
# def active_event(event_tuple):
#     # Events must be within 3 days of the current date at time of extraction
#     if abs((event_tuple[2] - datetime.datetime.now()).days) < 3:
#         return True
#     return False

# active_event(events), active_event(extract_events(r_objs["minnesota-twins"]))

In [328]:
r_objs.keys()

dict_keys(['minnesota-vikings', 'minnesota-twins', 'minnesota-timberwolves', 'minnesota-wild', 'minnesota-lynx', 'uconn-huskies'])

In [329]:
EVENTS = []

for team, r in r_objs.items():
    
    if r:
        EVENTS.append(Event.from_team_json(r))

In [314]:
my_event = Event('401697326', 'Minnesota Twins at Philadelphia Phillies', datetime.datetime(2025, 9, 28, 19, 5), [('Philadelphia Phillies', 2), ('Minnesota Twins', 1)])

In [315]:
my_event.__repr__()

'<Event 401697326 at 2025-09-28 19:05:00 named Minnesota Twins at Philadelphia Phillies>'

In [316]:
my_event.active()

False

In [317]:
my_event.to_dict()

{'id': '401697326',
 'name': 'Minnesota Twins at Philadelphia Phillies',
 'datetime': '2025-09-28T19:05:00',
 'future': False,
 'score': [('Philadelphia Phillies', 2), ('Minnesota Twins', 1)]}

In [318]:
json.dumps(my_event.to_dict())

'{"id": "401697326", "name": "Minnesota Twins at Philadelphia Phillies", "datetime": "2025-09-28T19:05:00", "future": false, "score": [["Philadelphia Phillies", 2], ["Minnesota Twins", 1]]}'

In [319]:
read_event = Event.from_dict(json.loads('{"id": "401697326", "name": "Minnesota Twins at Philadelphia Phillies", "datetime": "2025-09-28T19:05:00", "future": false, "score": [["Philadelphia Phillies", 2], ["Minnesota Twins", 1]]}'))

In [320]:
read_event.to_lines()

['Minnesota Twins at Philadelphia Phillies - Sun Sep 28',
 'Philadelphia Phillies: 2  |  Minnesota Twins: 1']

In [ ]:
# def build_trml_payload(data):
#     source_data = {x:data[x] for x in range(len(data))}
#     return {"merge_variables": source_data}
# build_trml_payload(data)

In [ ]:
# r = requests.post(webhook_url, headers={"Content-Type": "application/json"}, json=build_trml_payload(data))
# r.status_code